# mp-spawn-workers — worked example 2: Verify that a worker exception propagates to the parent process

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mp-spawn-workers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When a worker process launched with `mp.spawn` raises an exception, `spawn` re-raises it in the parent process as a `ProcessRaisedException` (or `Exception` in older PyTorch). This is an important safety property: a silent worker crash without the parent seeing an error would cause training to hang or produce wrong results. Testing that errors propagate gives you confidence the launcher will surface failures.

## Worked solution

**Step 1 — write a worker that raises on a specific rank.** We create a worker function that raises `RuntimeError` when `rank == 0`. This lets us test that the error is actually surfaced.

**Step 2 — call `mp.spawn(..., join=True)` inside a `try/except`.** We expect an exception to be raised. The `except` clause confirms the type and message.

**Step 3 — verify the exception message contains our sentinel string.** The original exception's message should be embedded in the parent-side re-raise, so we can check for a known substring.

**Step 4 — confirm no results if error occurred.** If the worker raised, the shared dict should be empty (rank 1 may not have written either, depending on timing).

**Why this matters.** In a real DDP training run, if the gradient reduction call fails on rank 1 (e.g., a NCCL timeout), you want the launcher to immediately raise in the main process rather than hanging indefinitely.

In [ ]:
import sys
import importlib
import multiprocessing
import torch.multiprocessing as mp

ERROR_WORKER_SRC = '''
def worker(rank, world_size, shared_dict):
    if rank == 0:
        raise RuntimeError("intentional_failure_rank0")
    shared_dict[rank] = rank  # rank 1 writes its value
'''

def test_error_propagation(port: int = 29502):
    with open('/tmp/dd_error_worker.py', 'w') as f:
        f.write(ERROR_WORKER_SRC)
    if '/tmp' not in sys.path:
        sys.path.insert(0, '/tmp')
    if 'dd_error_worker' in sys.modules:
        mod = importlib.reload(sys.modules['dd_error_worker'])
    else:
        mod = importlib.import_module('dd_error_worker')

    manager = multiprocessing.Manager()
    shared_dict = manager.dict()

    caught_error = None
    try:
        mp.spawn(mod.worker, args=(2, shared_dict), nprocs=2, join=True)
    except Exception as e:
        caught_error = e

    return caught_error

try:
    err = test_error_propagation()
    if err is not None:
        print(f"Caught expected exception: {type(err).__name__}")
        print(f"Contains sentinel: {'intentional_failure_rank0' in str(err)}")
    else:
        print("No exception raised — spawn not fully supported in this environment")
except Exception as outer:
    print(f"Outer exception (spawn unavailable): {outer}")
    print("Expected: exception containing 'intentional_failure_rank0'")